# ✅ เฉลย Challenge 08 — โมเดลทำนายมูลค่านักเตะ (Machine Learning เบื้องต้น)

**Week 08 · Machine Learning เบื้องต้น · 30 นาที**

## ภารกิจ
1. เตรียม Features ($X$) และ Target ($y = \text{value\_millions}$)
2. ทำความสะอาดค่าว่างหรือค่าข้อความใน `value_millions`
3. แบ่งชุดข้อมูล Train (80%) / Test (20%)
4. เทรนโมเดล `LinearRegression` และวัดผลด้วย MAE เทียบกับ Baseline
5. **ทำนายข้อมูลชุดใหม่ (Unseen Data):** ประเมินราคานักเตะดาวรุ่งจากไฟล์ `../data/scouted_players.csv`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# 1. โหลดข้อมูลหลัก
df = pd.read_csv("../data/footballers.csv")

# 2. Clean ข้อมูลราคา (แปลงข้อความ unknown ให้เป็น NaN แล้วตัดแถวว่างออก)
df["value_millions"] = pd.to_numeric(df["value_millions"], errors="coerce")
df_clean = df.dropna(subset=["value_millions", "age"]).copy()

print(f"จำนวนข้อมูลพร้อมฝึกโมเดล: {len(df_clean)} คน")

In [ ]:
# 3. กำหนด Features (X) และ Target (y)
features = ["overall", "age", "pace", "shooting", "passing"]
X = df_clean[features]
y = df_clean["value_millions"]

# 4. แบ่งข้อมูล Train / Test (80:20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"ขนาดชุดฝึก (Train): {X_train.shape[0]} ตัวอย่าง")
print(f"ขนาดชุดทดสอบ (Test): {X_test.shape[0]} ตัวอย่าง")

In [ ]:
# 5. เทรนโมเดล Linear Regression จากชุดฝึกเท่านั้น
model = LinearRegression()
model.fit(X_train, y_train)

# 6. วัดผลความแม่นยำบน Test Set
y_pred = model.predict(X_test)
model_mae = mean_absolute_error(y_test, y_pred)

# เปรียบเทียบกับ Baseline (การทายค่าเฉลี่ยของ y_train)
baseline_pred = [y_train.mean()] * len(y_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Baseline MAE (ทายค่าเฉลี่ย): {baseline_mae:.2f} ล้านยูโร")
print(f"Model MAE (Linear Regression): {model_mae:.2f} ล้านยูโร")
print(f"-> โมเดลมีความคลาดเคลื่อนลดลง {baseline_mae - model_mae:.2f} ล้านยูโร เมื่อเทียบกับ Baseline")

In [ ]:
# 7. นำโมเดลไปทำนายข้อมูลนักเตะชุดใหม่ที่ไม่เคยเห็น (Unseen Data)
new_df = pd.read_csv("../data/scouted_players.csv")
X_new = new_df[features]

# ประเมินราคาด้วยโมเดล
new_df["predicted_value_millions"] = model.predict(X_new).round(1)

print("=== ผลการประเมินราคานักเตะดาวรุ่งชุดใหม่ (Unseen Data) ===")
display(new_df[["name", "club", "age", "overall", "predicted_value_millions"]].sort_values("predicted_value_millions", ascending=False))

### 💡 คำเฉลย Concept Check

1. **การทำนายมูลค่านักเตะถือเป็น Machine Learning หรือไม่?**
   - **คำตอบ:** ถือเป็น Machine Learning โดยเป็น **Supervised Learning (การเรียนรู้แบบมีผู้สอน)** เพราะเรามีทั้งข้อมูลคุณลักษณะนักเตะ ($X$) และมูลค่าราคาจริงในอดีต ($y$) ให้โมเดลเรียนรู้
   - ลักษณะงานเป็น **Regression Task** เพราะเป้าหมายการทำนายเป็นค่าตัวเลขต่อเนื่อง (มูลค่าเงินล้านยูโร)

2. **ทำไมต้องแบ่ง Train/Test และทำไมการ Predict บน `scouted_players.csv` จึงสะท้อนการใช้งานจริง?**
   - **Train/Test Split:** ป้องกันไม่ให้โมเดล "จำข้อสอบ" (Overfitting) และใช้ Test set ในการวัดความแม่นยำอย่างเป็นกลาง
   - **Unseen Data Prediction:** การนำโมเดลไปใช้กับ `scouted_players.csv` คือหัวใจของ ML (Generalization) ซึ่งจำลองสถานการณ์จริงที่ทีมงาน/สโมสรต้องการนำโมเดลไปประเมินราคานักเตะดาวรุ่งที่ยังไม่มีราคาตลาดระบุไว้